# 00 · Setup and data

First notebook in the sequence.  Gets the environment working, pulls the
corpora, and builds the manifests everything else reads.

**Do this before anything else**, because two steps here have delays that will
otherwise block the whole team:

1. ASVspoof 2019 LA requires accepting terms on Edinburgh DataShare.
2. Kaggle GPU access needs a phone-verified account.

Order of notebooks:

| # | Notebook | Phase |
|---|---|---|
| 00 | setup and data | P0 |
| 01 | extract features | P0 |
| 02 | train baseline | P1 |
| 03 | train codec-robust | P2 |
| 04 | evaluate | P1–P2 |
| 05 | fine-tune | P7 |
| 06 | liveness analysis | P5 |
| 07 | Indian-language set | P6 |

In [ ]:
# --- Colab setup -----------------------------------------------------------
# Run this first in every notebook.  Idempotent.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    # Keep the repo and all caches on Drive so a disconnect does not cost you
    # the feature extraction pass.
    PROJECT = Path("/content/drive/MyDrive/voice-integrity")
    if not PROJECT.exists():
        raise SystemExit(
            f"Upload or clone the repo to {PROJECT} first.\n"
            "  !git clone <your-repo-url> /content/drive/MyDrive/voice-integrity"
        )
else:
    PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))

# Repo-local model cache.  Set BEFORE importing transformers, or it will use
# the default location and the cache will not be portable to the demo machine.
os.environ["HF_HOME"] = str(PROJECT / "cache" / "huggingface")
os.environ["TORCH_HOME"] = str(PROJECT / "cache" / "torch")
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

print("project:", PROJECT)
print("python :", sys.version.split()[0])

In [ ]:
if IN_COLAB:
    !pip install -q transformers speechbrain soundfile librosa pydantic pyyaml cryptography wandb
    !apt-get -qq install -y ffmpeg libopencore-amrnb-dev > /dev/null

# AMR-NB encoding is the one that silently goes missing.  If this prints
# nothing, your mobile-codec augmentation does nothing and the whole
# codec-robustness result quietly evaporates.
!ffmpeg -hide_banner -encoders 2>/dev/null | grep -i amr || echo "AMR-NB ENCODER MISSING"

In [ ]:
import torch
print("cuda available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device         :", torch.cuda.get_device_name(0))
    print("memory         : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## The P0 gate

Before any model exists, confirm the metric harness is trustworthy.  A random
classifier must report EER near 50%.  If it does not, every number produced
later is meaningless — and you would not find out for weeks.

In [ ]:
from vif.eval.metrics import sanity_check_random

result = sanity_check_random(n=20000)
print(result.summary())
assert 0.45 < result.eer < 0.55, "METRIC HARNESS IS BROKEN - stop and fix this"
print("\nP0 gate passed: the harness is trustworthy.")

## Corpora

Download these manually — the links are gated and cannot be scripted.

| Corpus | Purpose | Link |
|---|---|---|
| ASVspoof 2019 LA | training + in-domain eval | `datashare.ed.ac.uk/handle/10283/3336` |
| In-the-Wild | out-of-domain honesty eval | `deepfake-total.com/in_the_wild` |
| AI4Bharat Kathbath | Indian-language reference audio | `github.com/AI4Bharat/IndicSUPERB` |

Unpack under `data/raw/` so that `data/raw/LA/` contains the four ASVspoof
directories.

In [ ]:
from pathlib import Path

for name, path in [
    ("ASVspoof 2019 LA", Path("data/raw/LA")),
    ("In-the-Wild",      Path("data/raw/in_the_wild")),
    ("Kathbath",         Path("data/raw/kathbath")),
]:
    print(f"{name:<20} {'present' if path.exists() else 'MISSING'}  {path}")

## Manifests

Every corpus is reduced to one JSONL schema, so leave-one-attack-out,
per-codec and per-language slices become one-line filters instead of bespoke
scripts.

In [ ]:
import json
from vif.data.manifests import build_asvspoof19_la, summarise, write_manifest

manifest_dir = Path("data/manifests")
manifest_dir.mkdir(parents=True, exist_ok=True)

for split in ("train", "dev", "eval"):
    try:
        items = build_asvspoof19_la("data/raw/LA", split)
    except FileNotFoundError as exc:
        print(f"skip {split}: {exc}")
        continue
    write_manifest(items, manifest_dir / f"asvspoof19la_{split}.jsonl")
    print(f"\n=== {split} ===")
    print(json.dumps(summarise(items), indent=2))

### Read the imbalance line

`spoof_per_bonafide` will be roughly **9**.  That single number is why this
project never reports accuracy: a model that answers "spoof" unconditionally
scores about 90% and is useless.  Everything downstream reports **EER** and
**TPR at 1% FPR** instead.

In [ ]:
from vif.data.manifests import build_in_the_wild

try:
    wild = build_in_the_wild("data/raw/in_the_wild")
    write_manifest(wild, manifest_dir / "in_the_wild_eval.jsonl")
    print(json.dumps(summarise(wild), indent=2))
except FileNotFoundError as exc:
    print("In-the-Wild not present yet:", exc)

## Next

`01_extract_features.ipynb` — the one-time GPU pass that makes every later
experiment cost minutes instead of a day.